In [2]:
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import torch.autograd as ag

from collections import defaultdict, deque, namedtuple

In [3]:
# need code to read in dataset



In [4]:
# Some constants to keep around
LSTM_NUM_LAYERS = 1
TEST_EMBEDDING_DIM = 5
WORD_EMBEDDING_DIM = 64
STACK_EMBEDDING_DIM = 100
NUM_FEATURES = 3

# Hyperparameters
ETA_0 = 0.01
DROPOUT = 0.0

In [5]:
class ParserState:
    """
    Manages the state of a parse by keeping track of
    the input buffer and stack, and offering a public interface for
    doing actions (i.e, SHIFT, REDUCE_L, REDUCE_R)
    """

    def __init__(self, sentence, sentence_embs, combiner, null_stack_tok_embed=None):
        """
        :param sentence A list of strings, the words in the sentence
        :param sentence_embs A list of ag.Variable objects, where the ith element is the embedding
            of the ith word in the sentence
        :param combiner A network component that gives an output embedding given two input embeddings
            when doing a reduction
        :param null_stack_tok_embed ag.Variable The embedding of NULL_STACK_TOK
        """
        self.combiner = combiner

        # Input buffer is a list, along with an index into the list.
        # curr_input_buff_idx points to the *next element to pop off the input buffer*
        self.curr_input_buff_idx = 0
        self.input_buffer = [ StackEntry(we[0], pos, we[1]) for pos, we in enumerate(zip(sentence, sentence_embs)) ]

        self.stack = []
        self.null_stack_tok_embed = null_stack_tok_embed

    def shift(self):
        next_item = self.input_buffer[self.curr_input_buff_idx]
        self.stack.append(next_item)
        self.curr_input_buff_idx += 1

    def reduce_left(self):
        return self._reduce(Actions.REDUCE_L)

    def reduce_right(self):
        return self._reduce(Actions.REDUCE_R)

    def done_parsing(self):
        """
        Returns True if we are done parsing.
        Otherwise, returns False
        Remember that we are padding the input with an <END-OF-INPUT> token.
        self.curr_input_buff_idx is the just an index to the next token in the input buffer
        (i.e, each SHIFT increments this index by 1).
        <END-OF-INPUT> should not be shifted onto the stack ever.
        """
        # STUDENT
        pass
        # END STUDENT

    def stack_len(self):
        return len(self.stack)

    def input_buffer_len(self):
        return len(self.input_buffer) - self.curr_input_buff_idx

    def stack_peek_n(self, n):
        """
        Look at the top n items on the stack.
        If you ask for more than are on the stack, copies of the null_stack_tok_embed
        are returned
        :param n How many items to look at
        """
        if len(self.stack) - n < 0:
            return [ StackEntry(NULL_STACK_TOK, -1, self.null_stack_tok_embed) ] * (n - len(self.stack)) \
                   + self.stack[:]
        return self.stack[-n:]

    def input_buffer_peek_n(self, n):
        """
        Look at the next n words in the input buffer
        :param n How many words ahead to look
        """
        assert self.curr_input_buff_idx + n - 1 <= len(self.input_buffer)
        return self.input_buffer[self.curr_input_buff_idx:self.curr_input_buff_idx+n]

    def _reduce(self, action):
        """
        Reduce the top two items on the stack, combine them,
        and place the combination back on the stack.
        Combination means running the embeddings of the two stack items
        through your combiner network and getting a dense output.
        (The ParserState stores the combiner in the instance variable self.combiner).
        Important things to note:
            - Make sure you get which word is the head and which word
              is the modifer correct
            - Make sure that the order of the embeddings you pass into
              the combiner is correct.  The head word should always go first,
              and the modifier second.  Mixing it up will make it more difficult
              to learn the relationships it needs to learn
            - Make sure when creating the new dependency graph edge, that you store it in the DepGraphEdge object
              like this ( (head word, position of head word), (modifier word, position of modifier) ).
              Keeping track of the positions in the sentence is necessary to be able to uniquely
              identify edges when a sentence contains the same word multiple times.
              Technically the position is all that is needed, but it will be easier to debug
              if you carry along the word too.
        :param action Whether we reduce left or reduce right
        :return DepGraphEdge The edge that was formed in the dependency graph.
        """
        assert len(self.stack) >= 2, "ERROR: Cannot reduce with stack length less than 2"
        
        # STUDENT
        # hint: use list.pop()
        # END STUDENT

    def __str__(self):
        """
        Print the state for debugging
        """
        # only print the words, dont want to print the embeddings too
        return "Stack: {}\nInput Buffer: {}\n".format([ entry.headword for entry in self.stack ], 
                [ entry.headword for entry in self.input_buffer[self.curr_input_buff_idx:] ])


In [6]:
class DummyCombiner:

    def __call__(self, head, modifier):
        return head

In [7]:
test_sentence = "They can fish".split()

# These are what initialize the input buffer, and are used on the stack.
# headword: The head word, stored as a string
# headword_pos: The position of the headword in the sentence as an int
# embedding: The embedding of the phrase as an autograd.Variable
StackEntry = namedtuple("StackEntry", ["headword", "headword_pos", "embedding"])

In [8]:
parser_state = ParserState(
    test_sentence, [None] * len(test_sentence), DummyCombiner())

In [9]:
print(parser_state)

Stack: []
Input Buffer: ['They', 'can', 'fish']

